# 03. Comprehensive Model Validation & Stress Testing
**Project**: AI Model Risk & GenAI Evaluation Framework
**Focus**: Performance, Error Profiling, Fairlearn Bias Audit, SHAP, and Robustness


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import joblib

sys.path.insert(0, os.path.abspath('..'))
from src.performance import PerformanceEvaluator
from src.error_analysis import ErrorAnalyzer
from src.fairness import FairnessAuditor
from src.explainability import ModelExplainer
from src.robustness import RobustnessTester


## 1. Load Test Holdout Data & Champion Model


In [ ]:
test_df = pd.read_csv('../data/processed/test_split.csv')
champion_model = joblib.load('../models/credit_model.joblib')

feature_cols = [
    'annual_income', 'loan_amount', 'employment_length_years', 'credit_history_years',
    'num_open_credit_lines', 'debt_to_income_ratio', 'has_previous_default',
    'delinquent_2yrs', 'loan_purpose'
]

X_test = test_df[feature_cols]
y_test = test_df['risk_label']
y_prob = champion_model.predict_proba(X_test)[:, 1]
test_df['pred_prob'] = y_prob


## 2. Fair Lending Algorithmic Bias Audit (Fairlearn)


In [ ]:
auditor = FairnessAuditor(four_fifths_threshold=0.80)
bias_results = auditor.run_full_bias_audit(test_df)

print(f"Overall Fairness Status: {bias_results['overall_status']}")
for attr, res in bias_results['audited_attributes'].items():
    print(f"\nAttribute: {attr}")
    print(f"  Disparate Impact Ratio: {res['disparate_impact_ratio']}")
    print(f"  Status: {res['overall_governance_status']}")
    print(f"  Finding: {res['finding_summary']}")


## 3. Explainability & Reason Codes (SHAP)


In [ ]:
explainer = ModelExplainer(champion_model, background_data=X_test)
global_imp = explainer.compute_global_importance(X_test.iloc[:200])
print("=== Top 5 Global Risk Drivers ===")
print(global_imp.head(5))

print("\n=== Local Reason Code Explanation for Applicant #1 ===")
local_exp = explainer.explain_instance(X_test.iloc[0])
print(f"Predicted Default Probability: {local_exp['predicted_probability']:.1%}")
print("Top Adverse Factors (+Risk):", [f['feature'] for f in local_exp['top_risk_increasing_factors']])
print("Top Mitigating Factors (-Risk):", [f['feature'] for f in local_exp['top_risk_mitigating_factors']])


## 4. Robustness & Stress Resilience Testing


In [ ]:
robustness = RobustnessTester(champion_model)
rob_card = robustness.generate_robustness_scorecard(X_test, y_test)
print(f"Overall Robustness Status: {rob_card['overall_robustness_status']}")
print("\nMissing Data Stress Results:")
print(pd.DataFrame(rob_card['missing_data_results']))
print("\nNoise Perturbation Results:")
print(pd.DataFrame(rob_card['noise_perturbation_results']))
print(f"\nRecession Macro Stress Shock: {rob_card['macro_stress_shock']['narrative']}")
